# Score a local image with AutoMM

Copy the complete model artifact directory and a JPEG or PNG to this environment first. Use the same AutoMM and PyTorch versions as training. Start with a fresh Python kernel and disable network access to verify that the model loads and predicts from local files.

In [ ]:
import json
import os
from importlib.metadata import version
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

model_dir = Path("downloaded-model")
image_file = Path("example.jpg").resolve()
if not image_file.is_file():
    raise FileNotFoundError("Set image_file to a local JPEG or PNG.")
metadata = json.loads((model_dir / "model.json").read_text(encoding="utf-8"))
if metadata["schema_version"] != 1:
    raise ValueError("Unsupported model metadata version.")
if version("autogluon.multimodal") != metadata["autogluon_version"]:
    raise RuntimeError("Use the AutoMM version recorded in model.json.")

In [ ]:
import pandas as pd
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor.load(str(model_dir / metadata["predictor_path"]), verbosity=0)
predictor.set_num_gpus(0)
image_column = metadata["input_schema"]["image"]["column"]
features = pd.DataFrame({image_column: [str(image_file)]})
print("Prediction:", predictor.predict(features).iloc[0])
display(predictor.predict_proba(features))